Experiment Notebook
**Authors:** M. Vamsi Krishna, S. Sri Harsha  
**Affiliation:** KL University, Vaddeswaram  


---

### How to run
1. Runtime → Change runtime type → CPU (GPU is not needed)  
2. Run all cells top to bottom — do not skip cells  
3. Taiwan Credit needs a manual upload (Cell 6 will prompt you)  
4. Runtime depends on the Colab CPU allocation and enabled analyses; record the elapsed time from your own run.  

### What this notebook does
Runs the benchmark pipeline and supplementary analyses. The datasets, split seeds, missingness rates, and model families are defined in the configuration cells. Runtime and output files are recorded by the run itself; no fixed duration is promised.


In [ ]:
# Install the pinned versions recorded in the primary run manifest.
%pip install -q numpy==2.1.3 pandas==2.2.3 scipy==1.15.2 scikit-learn==1.6.1 xgboost==3.2.0 lightgbm==4.6.0 shap==0.51.0 matplotlib==3.10.3 seaborn==0.13.2
print("Pinned benchmark dependencies installed; restart the runtime if prompted.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings, time, os, json
from pathlib import Path
from copy import deepcopy
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import recall_score, accuracy_score
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from scipy.stats import spearmanr
import shap
warnings.filterwarnings("ignore")
np.random.seed(42)
print("Imports loaded; primary analysis uses descriptive summaries, not seed-based hypothesis tests.")
print(f"numpy={np.__version__}; pandas={pd.__version__}; sklearn={__import__('sklearn').__version__}")


In [ ]:
# Experiment configuration: fixed resampling seeds for reproducibility.
# These are repeated splits of the same finite datasets, not independent populations.
SEEDS = [42, 123, 456, 1, 7, 13, 21, 37, 55, 99]
MISS_RATES = [0.05, 0.10, 0.20]

# Descriptive reference only; not a validated operational threshold.
MG_MIN = 0.01

ALL_MODELS = ["LR", "RF", "XGB", "LGB", "MLP"]
TREE_MODELS = ["RF", "XGB", "LGB", "MLP"]
MAIN_DATASETS = ["Adult Income", "COMPAS", "Taiwan Credit"]

print("Config loaded.")
print(f"Seeds: {SEEDS}")
print(f"Masking rates: {MISS_RATES}")
print(f"Descriptive FMG reference: {MG_MIN:.3f}")


In [ ]:
# ── Dataset 1: Adult Income ──────────────────────────────────────────────────
# UCI Adult training split: 32,561 raw rows; '?' values are parsed as missing below.
# Sensitive attribute: sex. Focus group: female (g0).
# Sex-specific class rates are reported from the cleaned sample and are
# not interpreted as causal; the matched-subsample analysis is descriptive.

def load_adult(filepath=None):
    # Prefer a provided UCI archive/file so the exact input can be hashed and reused.
    # The ZIP's training member is adult.data; adult.test is not appended here.
    import zipfile
    url = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
           "adult/adult.data")
    if filepath is None and Path("adult.zip").is_file():
        filepath = Path("adult.zip")
    elif filepath is None and Path("adult.data").is_file():
        filepath = Path("adult.data")
    cols = ["age","workclass","fnlwgt","education","education-num",
            "marital-status","occupation","relationship","race","sex",
            "capital-gain","capital-loss","hours-per-week","native-country","income"]

    if filepath is not None and str(filepath).lower().endswith(".zip"):
        with zipfile.ZipFile(filepath) as archive:
            members = [name for name in archive.namelist() if name.lower().endswith("adult.data")]
            if not members:
                raise FileNotFoundError("adult.zip does not contain adult.data")
            df = pd.read_csv(archive.open(members[0]), header=None, names=cols,
                             na_values=["?", " ?"], skipinitialspace=True)
    else:
        df = pd.read_csv(filepath if filepath is not None else url, header=None, names=cols,
                         na_values=["?", " ?"], skipinitialspace=True)
    globals()["ADULT_RAW_ROWS"] = len(df)
    globals()["ADULT_MISSING_ROWS"] = int(df.isna().any(axis=1).sum())
    globals()["ADULT_SOURCE"] = str(filepath) if filepath is not None else url
    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Binary target
    df["label"] = (df["income"].str.strip().str.replace(".", "", regex=False) == ">50K").astype(int)
    df.drop(columns=["income"], inplace=True)

    # Encode sex → sensitive attribute
    # female=0 (focus group g0), male=1
    df["sensitive"] = (df["sex"].str.strip() == "Male").astype(int)
    df.drop(columns=["sex"], inplace=True)

    # Categorical encoding is performed after the train/test split in
    # preprocess(), so encoder mappings are fitted on training data only.

    feature_cols = [c for c in df.columns if c not in ("label", "sensitive")]

    print(f"Adult Income: {len(df)} cleaned rows, {len(feature_cols)} features")
    if len(df) != 30162:
        warnings.warn(f"Expected 30,162 cleaned rows; observed {len(df)}. Verify the Adult file/version.")
    print(f"  Female (g0): {(df.sensitive==0).sum()}  "
          f"  positive rate: {df[df.sensitive==0]['label'].mean():.3f}")
    print(f"  Male   (g1): {(df.sensitive==1).sum()}  "
          f"  positive rate: {df[df.sensitive==1]['label'].mean():.3f}")

    return df, feature_cols, "sensitive", "label"

adult_df, adult_feats, adult_sens, adult_tgt = load_adult()


In [ ]:
# ── Dataset 2: COMPAS ────────────────────────────────────────────────────────
# ProPublica's recidivism dataset; these filters yield 5,278 rows.
# Sensitive attribute: race. Focus group: African-American (g0).
# We follow standard preprocessing from the fairness literature.
# Ethics note is in the paper — not repeating here, but please read it.

def load_compas():
    url = ("https://raw.githubusercontent.com/propublica/compas-analysis/"
           "master/compas-scores-two-years.csv")
    df = pd.read_csv(url)

    # Standard ProPublica filtering (same as Chouldechova 2017)
    df = df[
        (df["days_b_screening_arrest"] >= -30) &
        (df["days_b_screening_arrest"] <= 30) &
        (df["is_recid"] != -1) &
        (df["c_charge_degree"] != "O") &
        (df["score_text"] != "N/A")
    ].copy()
    df.reset_index(drop=True, inplace=True)

    keep = ["age", "c_charge_degree", "race", "sex",
            "priors_count", "days_b_screening_arrest",
            "juv_fel_count", "juv_misd_count", "juv_other_count",
            "c_days_from_compas", "is_recid"]
    df = df[keep].dropna()

    df["label"] = df["is_recid"].astype(int)

    # sensitive: African-American=0 (g0), Caucasian=1 (g1)
    df["sensitive"] = (~(df["race"] == "African-American")).astype(int)

    # Only keep African-American and Caucasian for binary comparison
    df = df[df["race"].isin(["African-American", "Caucasian"])].copy()
    df.reset_index(drop=True, inplace=True)

    # Categorical encoding is performed after the train/test split in
    # preprocess(), so encoder mappings are fitted on training data only.

    # Exclude race from model features because it defines the COMPAS focus group.
    # This matches the sensitive-feature exclusion used for the other datasets.
    feature_cols = [c for c in df.columns
                    if c not in ("label", "sensitive", "is_recid", "race")]

    print(f"COMPAS: {len(df)} rows, {len(feature_cols)} features")
    print(f"  AfAm  (g0): {(df.sensitive==0).sum()}  "
          f"  positive rate: {df[df.sensitive==0]['label'].mean():.3f}")
    print(f"  Cauc  (g1): {(df.sensitive==1).sum()}  "
          f"  positive rate: {df[df.sensitive==1]['label'].mean():.3f}")

    return df, feature_cols, "sensitive", "label"

compas_df, compas_feats, compas_sens, compas_tgt = load_compas()


In [ ]:
# ── Dataset 3: Taiwan Credit ─────────────────────────────────────────────────
# 30000 rows, UCI dataset ID 350.
# This one can't be fetched programmatically due to URL instability
# on the UCI server, so I'm using a manual upload fallback.
# Download from: https://www.kaggle.com/datasets/uciml/default-of-credit-card-clients-dataset
# File name should be: UCI_Credit_Card.csv  (or similar)

from google.colab import files as colab_files

def load_taiwan(filepath=None):
    if filepath is None:
        print("Upload the Taiwan Credit CSV file now...")
        uploaded = colab_files.upload()
        filepath = list(uploaded.keys())[0]

    globals()["TAIWAN_SOURCE"] = str(filepath)
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

    # Some versions have an 'id' column, some don't
    if "id" in df.columns:
        df.drop(columns=["id"], inplace=True)

    # The target is default.payment.next.month
    # Column name varies between versions — find it
    target_col = [c for c in df.columns
                  if "default" in c or "payment" in c][-1]
    df["label"] = df[target_col].astype(int)
    if target_col != "label":
        df.drop(columns=[target_col], inplace=True)

    # sensitive: sex column — 1=male, 2=female in original encoding
    # We map: female=0 (g0), male=1 (g1)
    sex_col = [c for c in df.columns if "sex" in c][0]
    df["sensitive"] = (df[sex_col] == 1).astype(int)  # male=1 → g1
    df.drop(columns=[sex_col], inplace=True)

    feature_cols = [c for c in df.columns if c not in ("label", "sensitive")]
    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)

    print(f"Taiwan Credit: {len(df)} rows, {len(feature_cols)} features")
    print(f"  Female (g0): {(df.sensitive==0).sum()}  "
          f"  positive rate: {df[df.sensitive==0]['label'].mean():.3f}")
    print(f"  Male   (g1): {(df.sensitive==1).sum()}  "
          f"  positive rate: {df[df.sensitive==1]['label'].mean():.3f}")

    return df, feature_cols, "sensitive", "label"

taiwan_df, taiwan_feats, taiwan_sens, taiwan_tgt = load_taiwan()


In [ ]:
# ── Dataset 4: German Credit ─────────────────────────────────────────────────
# OpenML ID 31 is credit-g, the original Statlog German Credit data
# (1,000 rows, 20 predictors plus target), not the separate South German
# Credit dataset corrected by Grömping (2019). This dataset is exploratory.

from sklearn.datasets import fetch_openml

def load_german():
    print("Fetching German Credit from OpenML (ID=31)...")
    dataset = fetch_openml(data_id=31, as_frame=True, parser="auto")
    df = dataset.frame.copy()
    df.columns = df.columns.str.lower().str.replace("-","_")

    # OpenML credit-g uses the original A91–A95 personal_status codes
    # female: A92 (female divorced/separated/married) and A95 (female single)
    # male: A91, A93, A94
    if "personal_status" not in df.columns:
        raise ValueError("personal_status column missing — verify OpenML credit-g ID 31")

    # Quick sanity check — make sure we see female categories
    cats = df["personal_status"].unique()
    assert any("female" in str(c).lower() or "A92" in str(c) or "A95" in str(c)
               for c in cats),         f"personal_status categories look wrong: {cats}"

    # female=0 (g0 focus group), male=1
    df["sensitive"] = (~df["personal_status"].isin(["A92", "A95",
        "female div/dep/mar", "female : divorced/separated/married",
        "female single"])).astype(int)

    # Target: class column — "good" or "bad" credit
    df["label"] = (df["class"] == "good").astype(int)
    df.drop(columns=["class", "personal_status"], inplace=True)

    # Categorical encoding is performed after the train/test split in
    # preprocess(), so encoder mappings are fitted on training data only.
    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)

    feature_cols = [c for c in df.columns if c not in ("label","sensitive")]

    print(f"German Credit: {len(df)} rows, {len(feature_cols)} features")
    print(f"  Female (g0): {(df.sensitive==0).sum()}  "
          f"  positive rate: {df[df.sensitive==0]['label'].mean():.3f}")
    print(f"  Male   (g1): {(df.sensitive==1).sum()}  "
          f"  positive rate: {df[df.sensitive==1]['label'].mean():.3f}")

    return df, feature_cols, "sensitive", "label"

german_df, german_feats, german_sens, german_tgt = load_german()


In [ ]:
# Bundle everything into one dict so the main loop can iterate cleanly.
# German Credit results are always labelled exploratory.

DATASETS = {
    "Adult Income" : (adult_df,  adult_feats,  adult_sens,  adult_tgt,  False),
    "COMPAS"       : (compas_df, compas_feats, compas_sens, compas_tgt, False),
    "Taiwan Credit": (taiwan_df, taiwan_feats, taiwan_sens, taiwan_tgt, False),
    "German Credit": (german_df, german_feats, german_sens, german_tgt, True),
    # last bool = is_exploratory
}

for name, (df, feats, sens, tgt, expl) in DATASETS.items():
    label = "(exploratory)" if expl else ""
    print(f"  {name:15s} {len(df):6d} rows  {len(feats):2d} feats  {label}")


In [ ]:
# ── Core functions: preprocessing, stressor injection, evaluation ────────────

def preprocess(df, feats, target, seed):
    """
    Standard train/test split with stratification.
    Fit scaler on train only — no leakage.
    Preserves original row indices so subgroup labels stay aligned with
    the corresponding test labels and predictions.
    """
    X = df[feats].copy()
    y = df[target].values

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.30, random_state=seed, stratify=y
    )

    # Fit category mappings on training data only. Test-only categories
    # receive the reserved value -1 rather than affecting the mapping.
    categorical_cols = X_tr.select_dtypes(include=['object', 'category']).columns
    for col in categorical_cols:
        train_values = X_tr[col].astype(str)
        test_values = X_te[col].astype(str)
        encoder = LabelEncoder().fit(train_values)
        mapping = {value: code for code, value in enumerate(encoder.classes_)}
        X_tr[col] = train_values.map(mapping).astype(float)
        X_te[col] = test_values.map(mapping).fillna(-1).astype(float)

    scaler = MinMaxScaler()
    X_tr = pd.DataFrame(scaler.fit_transform(X_tr), columns=feats, index=X_tr.index)
    X_te = pd.DataFrame(scaler.transform(X_te),     columns=feats, index=X_te.index)

    if not X_tr.index.is_unique or not X_te.index.is_unique:
        raise ValueError("Train/test row indices must be unique for subgroup alignment.")
    if X_tr.index.intersection(X_te.index).size != 0:
        raise ValueError("Train and test indices overlap.")

    return X_tr, X_te, y_tr, y_te


def aligned_test_sensitive(df, X_te, sens_col, y_te):
    """Return test-set sensitive labels in exactly the X_te/y_te row order."""
    sens_te = df.loc[X_te.index, sens_col].to_numpy()
    if len(sens_te) != len(X_te) or len(sens_te) != len(y_te):
        raise ValueError("Sensitive labels, test features, and test targets are misaligned.")
    return sens_te


def inject_mcar(X, rate, seed):
    """
    Completely At Random missingness — each value independently set to NaN
    with probability `rate`, regardless of its value or any other variable.
    Applied to test set only, after the model is already trained.
    """
    rng = np.random.RandomState(seed)
    X_out = X.copy()
    mask = rng.rand(*X_out.shape) < rate
    X_out[mask] = np.nan
    return X_out


def inject_group_conditional_mar(X, rate, sensitive, seed):
    """Construct one group-conditional MAR stress test on test feature cells.

    Mask probability depends on the observed audit group, not on the value of
    the feature cell being masked. Probabilities preserve the requested
    expected overall cell-missingness rate while assigning a higher rate to
    g0. This is one constructed mechanism, not all MAR or MNAR processes.
    """
    sensitive = np.asarray(sensitive)
    if set(np.unique(sensitive)) != {0, 1}:
        raise ValueError("Both focus (0) and comparison (1) groups are required.")
    q0 = float(np.mean(sensitive == 0))
    q1 = 1.0 - q0
    delta = 0.5 * float(rate)
    p0 = float(rate) + delta * q1
    p1 = float(rate) - delta * q0
    if not (0 <= p0 <= 1 and 0 <= p1 <= 1):
        raise ValueError(f"Invalid calibrated probabilities: p0={p0}, p1={p1}")
    row_p = np.where(sensitive == 0, p0, p1)[:, None]
    rng = np.random.RandomState(int(seed))
    mask = rng.rand(*X.shape) < row_p
    X_masked = X.copy()
    X_masked[mask] = np.nan
    diagnostics = {
        "realized_overall_rate": float(mask.mean()),
        "realized_g0_rate": float(mask[sensitive == 0].mean()),
        "realized_g1_rate": float(mask[sensitive == 1].mean()),
        "expected_g0_rate": p0,
        "expected_g1_rate": p1,
    }
    return X_masked, diagnostics


def impute(X_tr, X_te, strategy="simple"):
    """
    Impute test set using statistics from the training set only.
    'simple' = median on the label-encoded, scaled feature matrix
    'knn'    = 5-nearest neighbours
    'mice'   = iterative imputer (MICE)
    """
    if strategy == "simple":
        imp = SimpleImputer(strategy="median")
    elif strategy == "knn":
        imp = KNNImputer(n_neighbors=5)
    elif strategy == "mice":
        imp = IterativeImputer(random_state=42, max_iter=10)
    else:
        raise ValueError(f"Unknown imputation strategy: {strategy}")

    imp.fit(X_tr)
    X_te_imp = pd.DataFrame(imp.transform(X_te), columns=X_te.columns, index=X_te.index)
    X_tr_imp = pd.DataFrame(imp.transform(X_tr), columns=X_tr.columns, index=X_tr.index)
    return X_tr_imp, X_te_imp


def get_models(seed):
    """
    All five classifier families with the exact hyperparameters used in the paper.
    Nothing exotic — just the defaults that sklearn/xgb/lgb ship with,
    except we cap iterations to keep runtime manageable.
    """
    return {
        "LR" : LogisticRegression(max_iter=1000, C=1.0,
                                   solver="lbfgs", random_state=seed),
        "RF" : RandomForestClassifier(n_estimators=100, random_state=seed,
                                       n_jobs=-1),
        "XGB": XGBClassifier(n_estimators=100, eval_metric="logloss",
                              random_state=seed, verbosity=0,
                              use_label_encoder=False),
        "LGB": LGBMClassifier(n_estimators=100, random_state=seed,
                               verbosity=-1),
        "MLP": MLPClassifier(hidden_layer_sizes=(50,30), max_iter=500,
                              early_stopping=True, random_state=seed),
    }


def evaluate(model, X_te, y_te, sensitive):
    """
    Returns accuracy and focus-group recall (recall for g0, i.e. sensitive==0).
    Also returns comparison-group recall (g1) for the EOG calculation.
    """
    y_pred = model.predict(X_te)

    acc = accuracy_score(y_te, y_pred)

    g0_mask = sensitive == 0
    g1_mask = sensitive == 1

    rec_g0 = recall_score(y_te[g0_mask], y_pred[g0_mask], zero_division=0)
    rec_g1 = recall_score(y_te[g1_mask], y_pred[g1_mask], zero_division=0)

    return {"accuracy": acc, "recall_g0": rec_g0, "recall_g1": rec_g1}


def masking_gap(m_clean, m_stressed):
    """
    MG = (recall_g0_clean - recall_g0_stressed) - (acc_clean - acc_stressed)

    Positive MG means positive-class recall fell more than aggregate accuracy.
    It does not establish that g0 lost more recall than g1; compare
    the two subgroup recall losses for a between-group statement.
    """
    recall_drop = m_clean["recall_g0"] - m_stressed["recall_g0"]
    acc_drop    = m_clean["accuracy"]  - m_stressed["accuracy"]
    return recall_drop - acc_drop

print("Core functions defined.")


In [ ]:
# ── Main experiment loop ─────────────────────────────────────────────────────
# Full benchmark; runtime varies by hardware and package versions.
# If it times out, the results dicts are serialised at the end
# so you can restart and reload.
#
# Structure:
#   for each dataset:
#     for each seed:
#       split + scale (seed controls the split)
#       train all 5 models on CLEAN training set
#       evaluate on CLEAN test set  (baseline)
#       for each missingness rate:
#         inject paired nested MCAR masks into test set (same masks shared across models)
#         impute with SimpleImputer (train statistics)
#         evaluate stressed models

print("Starting main experiment...")
print("Runtime varies by hardware; elapsed time is reported at completion.")
print("="*60)

all_results = []   # one row per (dataset, model, seed, rate)
start_time  = time.time()

for ds_name, (df, feats, sens_col, tgt_col, is_expl) in DATASETS.items():
    ds_start = time.time()
    print(f"\n{ds_name} {'(exploratory)' if is_expl else ''}")

    for seed in SEEDS:
        X_tr, X_te, y_tr, y_te = preprocess(df, feats, tgt_col, seed)

        sens_te = aligned_test_sensitive(df, X_te, sens_col, y_te)

        models = get_models(seed)

        for mdl_name, model in models.items():
            # Train on clean data — no stressor in training
            model.fit(X_tr, y_tr)

            # Clean baseline
            m_clean = evaluate(model, X_te, y_te, sens_te)
            clean_prediction_reference = model.predict(X_te).copy()

            # Clean = rate 0
            all_results.append({
                "dataset"    : ds_name,
                "model"      : mdl_name,
                "seed"       : seed,
                "rate"       : 0.0,
                "accuracy"   : m_clean["accuracy"],
                "recall_g0"  : m_clean["recall_g0"],
                "recall_g1"  : m_clean["recall_g1"],
                "masking_gap": 0.0,
                "exploratory": is_expl,
            })

            # Stressed: inject MCAR, impute, evaluate
            for rate in MISS_RATES:
                X_stressed = inject_mcar(X_te, rate, seed=seed)
                _, X_imp    = impute(X_tr, X_stressed, strategy="simple")

                m_stressed = evaluate(model, X_imp, y_te, sens_te)
                assert np.array_equal(clean_prediction_reference, model.predict(X_te)), \
                    'Clean-test predictions changed during masked evaluation.'
                mg = masking_gap(m_clean, m_stressed)

                all_results.append({
                    "dataset"    : ds_name,
                    "model"      : mdl_name,
                    "seed"       : seed,
                    "rate"       : rate,
                    "accuracy"   : m_stressed["accuracy"],
                    "recall_g0"  : m_stressed["recall_g0"],
                    "recall_g1"  : m_stressed["recall_g1"],
                    "masking_gap": mg,
                    "exploratory": is_expl,
                })

        print(f"  seed {seed:3d} done", end="  ", flush=True)

    elapsed = (time.time() - ds_start) / 60
    print(f"\n  {ds_name} finished in {elapsed:.1f} min")

total_min = (time.time() - start_time) / 60
print(f"\nTotal time: {total_min:.1f} minutes")
print(f"Total rows: {len(all_results)}")

results_df = pd.DataFrame(all_results)
print("\nresults_df shape:", results_df.shape)
print(results_df.head(3))


In [ ]:
# Descriptive seed-level summaries only; no seed-based hypothesis tests.
stat_rows = []
for ds in results_df["dataset"].unique():
    for mdl in ALL_MODELS:
        sub = results_df[(results_df["dataset"] == ds) & (results_df["model"] == mdl)]
        clean = sub[sub["rate"] == 0.00].sort_values("seed")
        stress = sub[sub["rate"] == 0.20].sort_values("seed")
        if len(stress) < 2:
            continue
        mg = stress["masking_gap"].to_numpy(dtype=float)
        acc_drop = clean["accuracy"].to_numpy() - stress["accuracy"].to_numpy()
        rec_drop = clean["recall_g0"].to_numpy() - stress["recall_g0"].to_numpy()
        is_main = (ds in MAIN_DATASETS) and (mdl in TREE_MODELS)
        stat_rows.append({
            "Dataset": ds, "Model": mdl,
            "Acc Drop mean": float(acc_drop.mean()), "Acc Drop SD": float(acc_drop.std(ddof=1)),
            "Recall Drop mean": float(rec_drop.mean()), "Recall Drop SD": float(rec_drop.std(ddof=1)),
            "FMG mean": float(mg.mean()), "FMG SD": float(mg.std(ddof=1)),
            "FMG min": float(mg.min()), "FMG max": float(mg.max()),
            "Positive FMG splits": int((mg > 0).sum()), "N splits": int(len(mg)),
            "Mean exceeds descriptive 0.01 reference": bool(mg.mean() > MG_MIN),
            "Main": is_main,
        })
stats_df = pd.DataFrame(stat_rows)
main = stats_df[stats_df["Main"]]
print(f"Primary model-dataset means above descriptive reference: {main['Mean exceeds descriptive 0.01 reference'].sum()}/12")
print(f"Mean FMG across 12 primary model-dataset pairs: {main['FMG mean'].mean():.4f}")


In [ ]:
# ── Relationship to Equal Opportunity ───────────────────────────────────────
# EOG = recall(g1) - recall(g0)
# Positive = g1 (comparison group) has higher recall
# EOG(stressed)-EOG(clean) equals recall-loss(g0)-recall-loss(g1).
# This signed contrast is reported beside FMG; FMG alone is not group-specific.

eog_rows = []
for ds in MAIN_DATASETS:
    for mdl in ALL_MODELS:
        sub = results_df[(results_df["dataset"]==ds) & (results_df["model"]==mdl)]

        eog_clean  = (sub[sub["rate"]==0.00]["recall_g1"].values -
                      sub[sub["rate"]==0.00]["recall_g0"].values)
        eog_stress = (sub[sub["rate"]==0.20]["recall_g1"].values -
                      sub[sub["rate"]==0.20]["recall_g0"].values)

        eog_rows.append({
            "Dataset"       : ds,
            "Model"         : mdl,
            "EOG Clean mean": eog_clean.mean(),
            "EOG Clean std" : eog_clean.std(),
            "EOG Stressed mean": eog_stress.mean(),
            "EOG Stressed std" : eog_stress.std(),
            "EOG Increase"  : eog_stress.mean() - eog_clean.mean(),
        })

eog_df = pd.DataFrame(eog_rows)
print("EOG Table (Adult Income):")
print(eog_df[eog_df["Dataset"]=="Adult Income"][
    ["Model","EOG Clean mean","EOG Stressed mean","EOG Increase"]].to_string(index=False))


In [ ]:
# ── Figure 1: Masking Gap curves ────────────────────────────────────────────
# 4-panel plot, one per dataset, all 5 models.
# Error bars = ±1 SD across 10 seeds.

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
palette = ["#2196F3","#F44336","#4CAF50","#9C27B0","#FF9800"]  # LR RF XGB LGB MLP
ds_order = ["Adult Income","COMPAS","Taiwan Credit","German Credit"]

for idx, ds in enumerate(ds_order):
    ax = axes[idx]
    sub = results_df[results_df["dataset"]==ds]

    for midx, mdl in enumerate(ALL_MODELS):
        m_sub = sub[sub["model"]==mdl]
        grp = m_sub.groupby("rate")["masking_gap"].agg(["mean","std"])
        x   = grp.index.values * 100
        ax.errorbar(x, grp["mean"], yerr=grp["std"],
                    marker="o", label=mdl, color=palette[midx],
                    capsize=4, linewidth=1.8, markersize=5)

    ax.axhline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.6)
    ax.set_title(ds, fontsize=12, fontweight="bold")
    ax.set_xlabel("MCAR Missingness Rate (%)", fontsize=10)
    ax.set_ylabel("Masking Gap", fontsize=10)
    ax.legend(fontsize=8, framealpha=0.7)
    ax.yaxis.grid(True, linestyle="--", alpha=0.35)
    ax.set_axisbelow(True)

    if ds == "German Credit":
        ax.text(0.04, 0.05, "Exploratory — excluded from main claims",
                transform=ax.transAxes, fontsize=7,
                color="#8B4513", style="italic",
                bbox=dict(boxstyle="round,pad=0.2", facecolor="#FFF3CD", alpha=0.8))

fig.suptitle("Figure 1: Masking Gap vs MCAR Missingness Rate\n(mean ± 1 SD, 10 seeds)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fig1_masking_gap_curves.png", bbox_inches="tight", dpi=150)
plt.savefig("fig1_masking_gap_curves.pdf", bbox_inches="tight", dpi=300)
plt.show()
print("Figure 1 saved.")


In [ ]:
# ── Figure 2: Accuracy vs Recall degradation bar chart ──────────────────────
# This is the most viscerally persuasive figure in the paper.
# Side-by-side: tiny blue bar (accuracy drop) vs huge red bar (recall drop).
# Taiwan RF's archived values were 0.0042 accuracy loss and 0.0854 g0 recall loss;
# this plot is regenerated from the current run rather than hard-coded.

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
ds_order_main = ["Adult Income","COMPAS","Taiwan Credit"]

for idx, ds in enumerate(ds_order_main):
    ax   = axes[idx]
    sub  = results_df[(results_df["dataset"]==ds)&(results_df["rate"]==0.20)]
    sub0 = results_df[(results_df["dataset"]==ds)&(results_df["rate"]==0.00)]

    x = np.arange(len(TREE_MODELS))
    w = 0.38

    for mi, mdl in enumerate(TREE_MODELS):
        acc_drop = (sub0[sub0["model"]==mdl]["accuracy"].values -
                    sub[sub["model"]==mdl]["accuracy"].values)
        rec_drop = (sub0[sub0["model"]==mdl]["recall_g0"].values -
                    sub[sub["model"]==mdl]["recall_g0"].values)

        ax.bar(mi - w/2, acc_drop.mean(), w, yerr=acc_drop.std(),
               color="#5B9BD5", capsize=4, label="Accuracy drop" if mi==0 else "")
        ax.bar(mi + w/2, rec_drop.mean(), w, yerr=rec_drop.std(),
               color="#E06C75", capsize=4, label="Recall drop (g0)" if mi==0 else "")

    ax.set_xticks(x)
    ax.set_xticklabels(TREE_MODELS, fontsize=10)
    ax.set_xlabel("Model", fontsize=10)
    ax.set_ylabel("Absolute drop at 20% MCAR", fontsize=9)
    ax.set_title(ds, fontsize=11, fontweight="bold")
    ax.yaxis.grid(True, linestyle="--", alpha=0.35)
    ax.set_axisbelow(True)
    if idx == 0:
        ax.legend(fontsize=9, loc="upper right")

fig.suptitle("Figure 2: Accuracy vs Focus-Group Recall Degradation at 20% MCAR",
             fontsize=12)
plt.tight_layout()
plt.savefig("fig2_acc_vs_recall.png", bbox_inches="tight", dpi=150)
plt.savefig("fig2_acc_vs_recall.pdf", bbox_inches="tight", dpi=300)
plt.show()
print("Figure 2 saved.")


In [ ]:
# ── SHAP analysis ────────────────────────────────────────────────────────────
# Only RF on Adult Income and COMPAS (both have the most interesting patterns).
# 500 subsampled test instances — full test set takes ~20 min per run.
# Comparing SHAP values under clean vs 20% MCAR to see which features shift.

SHAP_DS = ["Adult Income","COMPAS"]
SHAP_SUBSAMPLE = 500
shap_rows = []

for ds_name in SHAP_DS:
    df, feats, sens_col, tgt_col, _ = DATASETS[ds_name]
    seed = 42  # use seed 42 for reproducibility of SHAP

    X_tr, X_te, y_tr, y_te = preprocess(df, feats, tgt_col, seed)
    sens_te = aligned_test_sensitive(df, X_te, sens_col, y_te)

    rf = get_models(seed)["RF"]
    rf.fit(X_tr, y_tr)

    # Subsample for speed
    idx_sub = np.random.choice(len(X_te), SHAP_SUBSAMPLE, replace=False)
    X_sub   = X_te.iloc[idx_sub]
    s_sub   = sens_te[idx_sub]

    # SHAP on clean
    explainer   = shap.TreeExplainer(rf)
    shap_clean  = explainer.shap_values(X_sub)
    if isinstance(shap_clean, list):  # binary output → class 1
        shap_clean = shap_clean[1]

    # SHAP on stressed (20% MCAR, then imputed)
    X_stressed  = inject_mcar(X_sub, 0.20, seed=seed)
    _, X_imp    = impute(X_tr, X_stressed, strategy="simple")
    shap_stress = explainer.shap_values(X_imp)
    if isinstance(shap_stress, list):
        shap_stress = shap_stress[1]

    # Per-subgroup mean SHAP shift
    for gi, gname in [(0, "g0"), (1, "g1")]:
        mask = s_sub == gi
        # Take mean across observations to get feature importance vector (1D)
        mean_clean  = np.abs(shap_clean[mask]).mean(axis=0)
        mean_stress = np.abs(shap_stress[mask]).mean(axis=0)

        # Ensure we are dealing with 1D arrays for the row calculation
        if mean_clean.ndim > 1: mean_clean = mean_clean.mean(axis=-1)
        if mean_stress.ndim > 1: mean_stress = mean_stress.mean(axis=-1)

        shift = mean_stress - mean_clean

        for fi, feat in enumerate(feats):
            shap_rows.append({
                "dataset"   : ds_name,
                "feature"   : feat,
                "subgroup"  : gi,
                "subgroup_name": gname,
                "shap_clean": mean_clean[fi],
                "shap_stress": mean_stress[fi],
                "shap_shift": shift[fi],
            })

    # Feature ranking stability (Spearman on mean |SHAP|)
    # Collapse any remaining class dimension to ensure 1D input for spearmanr
    cr = np.abs(shap_clean).mean(axis=0)
    sr = np.abs(shap_stress).mean(axis=0)
    if cr.ndim > 1: cr = cr.mean(axis=-1)
    if sr.ndim > 1: sr = sr.mean(axis=-1)

    rho = spearmanr(cr, sr).statistic
    print(f"{ds_name} RF SHAP rank association (descriptive): rho={rho:.3f}")

shap_df = pd.DataFrame(shap_rows)
print(f"\nSHAP rows: {len(shap_df)}")

In [ ]:
# ── Figure 6: Per-subgroup SHAP shifts ──────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
TOP_N = 6

for idx, ds_name in enumerate(SHAP_DS):
    ax  = axes[idx]
    sub = shap_df[shap_df["dataset"]==ds_name]

    # Top N features by between-group asymmetry
    asym = sub.groupby("feature").apply(
        lambda x: abs(x[x.subgroup==0]["shap_shift"].values[0] -
                      x[x.subgroup==1]["shap_shift"].values[0])
        if len(x[x.subgroup==0]) and len(x[x.subgroup==1]) else 0
    ).nlargest(TOP_N).index.tolist()

    sub_top = sub[sub["feature"].isin(asym)].copy()
    y_pos   = {f: i for i, f in enumerate(asym)}

    h = 0.35
    for _, row in sub_top.iterrows():
        yi = y_pos[row["feature"]]
        if row["subgroup"] == 0:
            ax.barh(yi + h/2, row["shap_shift"], h,
                    color="#E06C75", alpha=0.85, label="g0 (focus)" if yi==0 else "")
        else:
            ax.barh(yi - h/2, row["shap_shift"], h,
                    color="#82B0D2", alpha=0.85, label="g1 (comparison)" if yi==0 else "")

    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_yticks(range(TOP_N))
    ax.set_yticklabels(asym, fontsize=9)
    ax.set_xlabel("SHAP shift (stressed − clean)", fontsize=9)
    ax.set_ylabel("Feature", fontsize=9)
    ax.set_title(f"{ds_name}\n(top {TOP_N} by between-group asymmetry)",
                 fontsize=10, fontweight="bold")
    ax.legend(fontsize=8)
    ax.xaxis.grid(True, linestyle="--", alpha=0.35)
    ax.set_axisbelow(True)

fig.suptitle("Figure 6: Per-Subgroup SHAP Feature Shifts at 20% MCAR (RF)",
             fontsize=12)
plt.tight_layout()
plt.savefig("fig6_shap_shifts.png", bbox_inches="tight", dpi=150)
plt.savefig("fig6_shap_shifts.pdf", bbox_inches="tight", dpi=300)
plt.show()
print("Figure 6 saved.")


In [ ]:
# ── Imputation ablation ──────────────────────────────────────────────────────
# RF only, seed 42 only. This is acknowledged as single-seed and illustrative.
# Three strategies: Simple median, KNN, MICE.
# On two datasets: Adult Income and COMPAS.

ablation_rows = []
seed = 42
ABL_DATASETS = ["Adult Income","COMPAS"]

for ds_name in ABL_DATASETS:
    df, feats, sens_col, tgt_col, _ = DATASETS[ds_name]
    X_tr, X_te, y_tr, y_te = preprocess(df, feats, tgt_col, seed)
    sens_te = aligned_test_sensitive(df, X_te, sens_col, y_te)

    rf = get_models(seed)["RF"]
    rf.fit(X_tr, y_tr)
    m_clean = evaluate(rf, X_te, y_te, sens_te)

    for rate in [0.10, 0.20]:
        X_stressed = inject_mcar(X_te, rate, seed=seed)

        for strategy in ["simple","knn","mice"]:
            _, X_imp = impute(X_tr, X_stressed, strategy=strategy)
            m_st = evaluate(rf, X_imp, y_te, sens_te)
            mg   = masking_gap(m_clean, m_st)

            ablation_rows.append({
                "dataset"  : ds_name,
                "imputer"  : strategy.upper() if strategy != "simple" else "Simple",
                "rate"     : rate,
                "masking_gap": mg,
                "accuracy" : m_st["accuracy"],
                "recall_g0": m_st["recall_g0"],
            })

ablation_df = pd.DataFrame(ablation_rows)
print("Imputation ablation results (Adult Income):")
print(ablation_df[ablation_df["dataset"]=="Adult Income"].to_string(index=False))


In [ ]:
# ── Figure 7: Imputation ablation bar chart ──────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = {"Simple":"#5B9BD5","KNN":"#E06C75","MICE":"#4CAF50"}

for idx, ds_name in enumerate(ABL_DATASETS):
    ax  = axes[idx]
    sub = ablation_df[ablation_df["dataset"]==ds_name]

    rates     = [0.10, 0.20]
    strategies= ["Simple","KNN","MICE"]
    x         = np.arange(len(rates))
    n         = len(strategies)
    w         = 0.22
    offsets   = np.linspace(-(n-1)/2, (n-1)/2, n) * w

    for si, strat in enumerate(strategies):
        vals = [sub[(sub["imputer"]==strat)&(sub["rate"]==r)]["masking_gap"].values[0]
                for r in rates]
        bars = ax.bar(x + offsets[si], vals, w,
                      color=colors[strat], label=strat, alpha=0.88)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=7.5)

    ax.set_xticks(x)
    ax.set_xticklabels(["10%","20%"], fontsize=10)
    ax.set_xlabel("Missingness rate", fontsize=10)
    ax.set_ylabel("Masking Gap", fontsize=10)
    ax.set_title(f"{ds_name}  (RF, seed 42)", fontsize=10, fontweight="bold")
    ax.yaxis.grid(True, linestyle="--", alpha=0.35)
    ax.set_axisbelow(True)
    ax.legend(fontsize=9)

fig.suptitle("Figure 7: Imputation Ablation — Masking Gap by Strategy (RF, seed 42)",
             fontsize=12)
plt.tight_layout()
plt.savefig("fig7_imputation_ablation.png", bbox_inches="tight", dpi=150)
plt.savefig("fig7_imputation_ablation.pdf", bbox_inches="tight", dpi=300)
plt.show()
print("Figure 7 saved.")


In [ ]:
# Expanded group-conditional MAR versus MCAR sensitivity.
# This matches the mechanism in the dedicated executed sensitivity notebook.
# Repeated seeds are benchmark splits, not independent populations.
MAR_DS = ["Adult Income", "COMPAS", "Taiwan Credit"]
MAR_MODELS = ["RF", "XGB", "LGB", "MLP"]
MAR_RATES = [0.10, 0.20]
mar_rows = []

for ds_name in MAR_DS:
    df, feats, sens_col, tgt_col, _ = DATASETS[ds_name]
    print(f"\\nGroup-conditional MAR: {ds_name}")
    for seed in SEEDS:
        X_tr, X_te, y_tr, y_te = preprocess(df, feats, tgt_col, seed)
        sens_te = aligned_test_sensitive(df, X_te, sens_col, y_te)
        for model_name in MAR_MODELS:
            model = get_models(seed)[model_name]
            model.fit(X_tr, y_tr)
            clean_prediction_reference = model.predict(X_te).copy()
            m_clean = evaluate(model, X_te, y_te, sens_te)
            for rate in MAR_RATES:
                X_mcar = inject_mcar(X_te, rate, seed=seed)
                _, X_mcar_imp = impute(X_tr, X_mcar, strategy="simple")
                m_mcar = evaluate(model, X_mcar_imp, y_te, sens_te)
                X_mar, diagnostics = inject_group_conditional_mar(X_te, rate, sens_te, seed)
                _, X_mar_imp = impute(X_tr, X_mar, strategy="simple")
                m_mar = evaluate(model, X_mar_imp, y_te, sens_te)
                assert np.array_equal(clean_prediction_reference, model.predict(X_te)), (
                    "Clean-test predictions changed during masked evaluation.")
                mg_mcar = masking_gap(m_clean, m_mcar)
                mg_mar = masking_gap(m_clean, m_mar)
                mar_rows.append({
                    "dataset": ds_name, "model": model_name, "seed": int(seed),
                    "rate_nominal": float(rate), "mg_mcar": float(mg_mcar),
                    "mg_mar": float(mg_mar),
                    "mar_minus_mcar_mg": float(mg_mar - mg_mcar),
                    "accuracy_drop_mcar": float(m_clean["accuracy"] - m_mcar["accuracy"]),
                    "accuracy_drop_mar": float(m_clean["accuracy"] - m_mar["accuracy"]),
                    "recall_g0_drop_mcar": float(m_clean["recall_g0"] - m_mcar["recall_g0"]),
                    "recall_g0_drop_mar": float(m_clean["recall_g0"] - m_mar["recall_g0"]),
                    "focus_positive_denominator": int(((sens_te == 0) & (y_te == 1)).sum()),
                    **diagnostics,
                })
    print("  completed all benchmark split seeds")

mar_df = pd.DataFrame(mar_rows)
mar_stats_df = mar_df.groupby(["dataset", "model", "rate_nominal"], sort=True).agg(
    n_splits=("seed", "size"),
    mean_mg_mcar=("mg_mcar", "mean"),
    mean_mg_mar=("mg_mar", "mean"),
    mean_mar_minus_mcar_mg=("mar_minus_mcar_mg", "mean"),
    sd_mar_minus_mcar_mg=("mar_minus_mcar_mg", "std"),
    mean_realized_overall_rate=("realized_overall_rate", "mean"),
    mean_realized_g0_rate=("realized_g0_rate", "mean"),
    mean_realized_g1_rate=("realized_g1_rate", "mean"),
).reset_index()
mar_stats_df["positive_split_differences"] = mar_df.groupby(
    ["dataset", "model", "rate_nominal"], sort=True
)["mar_minus_mcar_mg"].apply(lambda x: int((x > 0).sum())).to_numpy()
assert len(mar_df) == len(MAR_DS) * len(MAR_MODELS) * len(SEEDS) * len(MAR_RATES)
assert len(mar_stats_df) == len(MAR_DS) * len(MAR_MODELS) * len(MAR_RATES)
display(mar_stats_df)


In [ ]:
# Figure 3: group-conditional MAR minus MCAR paired contrast.
# Error bars show ±1 SD across the ten repeated benchmark splits.
dataset_colors = {"Adult Income": "#1976D2", "COMPAS": "#E65100", "Taiwan Credit": "#009688"}
model_markers = {"RF": "o", "XGB": "s", "LGB": "^", "MLP": "D"}
dataset_order = ["Adult Income", "COMPAS", "Taiwan Credit"]
model_order = ["RF", "XGB", "LGB", "MLP"]
fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=True)
for ax, rate in zip(axes, MAR_RATES):
    sub = mar_stats_df[mar_stats_df["rate_nominal"] == rate]
    y_positions, y_labels = [], []
    for di, ds_name in enumerate(dataset_order):
        for mi, model_name in enumerate(model_order):
            row = sub[(sub["dataset"] == ds_name) & (sub["model"] == model_name)].iloc[0]
            y = di * (len(model_order) + 0.7) + mi
            ax.errorbar(row["mean_mar_minus_mcar_mg"], y,
                        xerr=row["sd_mar_minus_mcar_mg"],
                        fmt=model_markers[model_name], color=dataset_colors[ds_name],
                        capsize=3, markersize=7, linewidth=1.6)
            y_positions.append(y)
            y_labels.append(model_name)
        if di < len(dataset_order) - 1:
            ax.axhline((di + 1) * (len(model_order) + 0.7) - 0.35,
                       color="#BDBDBD", linewidth=0.8)
    ax.axvline(0, color="#424242", linewidth=1)
    ax.set_yticks(y_positions, y_labels)
    ax.invert_yaxis()
    ax.set_xlabel("Paired change in FMG (group-conditional − MCAR)")
    ax.set_title(f"{int(rate*100)}% nominal missingness")
    ax.grid(axis="x", alpha=0.25)
    for di, ds_name in enumerate(dataset_order):
        y = di * (len(model_order) + 0.7) + (len(model_order)-1)/2
        ax.text(-0.15, y, ds_name, transform=ax.get_yaxis_transform(),
                ha="right", va="center", color=dataset_colors[ds_name], fontsize=9)
axes[0].set_ylabel("Classifier (repeated splits nested within dataset)")
fig.suptitle("Figure 3: Group-conditional missingness sensitivity\\n"
             "Mean paired contrast ±1 SD across ten seeds")
fig.text(0.5, 0.01,
         "Positive values indicate larger FMG under this constructed group-conditional mechanism than under MCAR.",
         ha="center", fontsize=9)
plt.tight_layout(rect=[0.08, 0.04, 1, 0.92])
plt.savefig("fig3_mar_vs_mcar.png", bbox_inches="tight", dpi=180)
plt.savefig("fig3_mar_vs_mcar.pdf", bbox_inches="tight", dpi=300)
plt.show()
print("Figure 3 saved from group-conditional results.")


In [ ]:
# Matched-subsample class-rate sensitivity: descriptive only.
# This one subsample comparison cannot isolate a causal class-imbalance effect.
matched_rows = []
df, feats, sens_col, tgt_col, _ = DATASETS["Adult Income"]
for seed in SEEDS:
    g0_pos=df[(df[sens_col]==0)&(df[tgt_col]==1)]; g0_neg=df[(df[sens_col]==0)&(df[tgt_col]==0)]
    g1_pos=df[(df[sens_col]==1)&(df[tgt_col]==1)]; g1_neg=df[(df[sens_col]==1)&(df[tgt_col]==0)]
    target_pos_rate=len(g0_pos)/(len(g0_pos)+len(g0_neg))
    n_g1_pos=len(g1_pos); n_g1_neg=min(int(n_g1_pos/target_pos_rate)-n_g1_pos,len(g1_neg))
    g1_neg_sample=g1_neg.sample(n=n_g1_neg,random_state=seed)
    balanced_df=pd.concat([g0_pos,g0_neg,g1_pos,g1_neg_sample]).sample(frac=1,random_state=seed).reset_index(drop=True)
    X_tr,X_te,y_tr,y_te=preprocess(balanced_df,feats,tgt_col,seed)
    sens_te=aligned_test_sensitive(balanced_df,X_te,sens_col,y_te)
    rf=get_models(seed)["RF"]; rf.fit(X_tr,y_tr)
    m_clean=evaluate(rf,X_te,y_te,sens_te)
    X_stressed=inject_mcar(X_te,0.20,seed=seed); _,X_imp=impute(X_tr,X_stressed,strategy="simple")
    matched_rows.append({"seed":seed,"masking_gap":masking_gap(m_clean,evaluate(rf,X_imp,y_te,sens_te))})
matched_df=pd.DataFrame(matched_rows)
orig_mg=results_df[(results_df.dataset=="Adult Income")&(results_df.model=="RF")&(results_df.rate==0.20)]["masking_gap"].to_numpy()
print(f"Original: mean FMG={orig_mg.mean():.4f}; positive splits={(orig_mg>0).sum()}/{len(orig_mg)}")
print(f"Matched: mean FMG={matched_df.masking_gap.mean():.4f}; positive splits={(matched_df.masking_gap>0).sum()}/{len(matched_df)}")
print("Descriptive comparison only; does not identify a causal class-imbalance effect.")


In [ ]:
# ── Figure 4: Class imbalance matched-subsample ──────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left panel: original vs balanced per seed
ax = axes[0]
x  = range(len(SEEDS))
ax.plot(x, orig_mg, "o-", color="#E06C75", linewidth=1.8,
        markersize=6, label=f"Original (imbalanced)")
ax.plot(x, matched_df["masking_gap"].values, "s--", color="#5B9BD5",
        linewidth=1.8, markersize=6, label="Matched-subsample (balanced)")
ax.axhline(orig_mg.mean(),    color="#E06C75", linewidth=1.2, linestyle=":",
           alpha=0.7, label=f"Orig mean={orig_mg.mean():.3f}")
ax.axhline(matched_df["masking_gap"].mean(), color="#5B9BD5", linewidth=1.2,
           linestyle=":", alpha=0.7,
           label=f"Balanced mean={matched_df['masking_gap'].mean():.3f}")
ax.axhline(0, color="black", linewidth=0.7, linestyle=":")
ax.set_xticks(x)
ax.set_xticklabels([str(s) for s in SEEDS], fontsize=8, rotation=30)
ax.set_xlabel("Seed", fontsize=9)
ax.set_ylabel("Masking Gap at 20% MCAR", fontsize=9)
ax.set_title("Adult Income RF — Original vs Balanced\n"
             f"Original: MG={orig_mg.mean():.3f} p={p_orig:.4f} | "
             f"Balanced: MG={matched_df['masking_gap'].mean():.3f} p={p_matched:.4f}",
             fontsize=9, fontweight="bold")
ax.legend(fontsize=7, loc="lower right")
ax.yaxis.grid(True, linestyle="--", alpha=0.35)

# Right panel: MG vs class rate differential
ax = axes[1]
def positive_rate_gap(dataset_name):
    frame, _, sensitive, target, *_ = DATASETS[dataset_name]
    rates = frame.groupby(sensitive)[target].mean()
    return float(rates.loc[1] - rates.loc[0])

ds_points = {
    name: (positive_rate_gap(name),
           results_df[(results_df["dataset"]==name)&
                      (results_df["model"]=="RF")&
                      (results_df["rate"]==0.20)]["masking_gap"].mean(),
           results_df[(results_df["dataset"]==name)&
                      (results_df["model"]=="RF")&
                      (results_df["rate"]==0.20)]["masking_gap"].std())
    for name in ("Adult Income", "COMPAS", "Taiwan Credit")
}
colors_scatter = {"Adult Income":"#E06C75","COMPAS":"#5B9BD5","Taiwan Credit":"#4CAF50"}

for ds_name, (diff, mg_m, mg_s) in ds_points.items():
    ax.scatter(diff, mg_m, s=120, color=colors_scatter[ds_name], zorder=3,
               label=f"{ds_name} (gap={diff:+.3f})")
    ax.errorbar(diff, mg_m, yerr=mg_s, fmt="none",
                color=colors_scatter[ds_name], capsize=5, linewidth=1.5)

ax.axhline(0, color="black", linewidth=0.7, linestyle=":")
ax.axvline(0, color="gray", linewidth=0.7, linestyle="--", alpha=0.5)
ax.set_xlabel("Positive class rate: g1 − g0", fontsize=9)
ax.set_ylabel("Masking Gap (RF, 20% MCAR)", fontsize=9)
ax.set_title("MG vs Class Rate Differential\nCOMPAS: g0 has higher rate, yet MG > 0",
             fontsize=9, fontweight="bold")
ax.legend(fontsize=7)
ax.yaxis.grid(True, linestyle="--", alpha=0.35)
ax.set_axisbelow(True)

fig.suptitle("Figure 4: Class Imbalance Matched-Subsample Analysis", fontsize=12)
plt.tight_layout()
plt.savefig("fig4_class_imbalance.png", bbox_inches="tight", dpi=150)
plt.savefig("fig4_class_imbalance.pdf", bbox_inches="tight", dpi=300)
plt.show()
print("Figure 4 saved.")


In [ ]:
# Post-hoc n=30 COMPAS extension: descriptive stability summary only.
SEEDS_30 = SEEDS + list(range(200,220))
assert len(SEEDS_30)==30
n30_rows=[]
df_c,feats_c,sens_c,tgt_c,_=DATASETS["COMPAS"]
for seed in SEEDS_30:
    X_tr,X_te,y_tr,y_te=preprocess(df_c,feats_c,tgt_c,seed)
    sens_te=aligned_test_sensitive(df_c,X_te,sens_c,y_te)
    for mdl_name in ["RF","MLP"]:
        model=get_models(seed)[mdl_name]; model.fit(X_tr,y_tr)
        clean=evaluate(model,X_te,y_te,sens_te)
        X_stressed=inject_mcar(X_te,0.20,seed=seed); _,X_imp=impute(X_tr,X_stressed,strategy="simple")
        mg=masking_gap(clean,evaluate(model,X_imp,y_te,sens_te))
        n30_rows.append({"dataset":"COMPAS","model":mdl_name,"seed":seed,"masking_gap":mg})
    print(f"seed {seed}",end=" ",flush=True)
n30_df=pd.DataFrame(n30_rows)
for mdl in ["RF","MLP"]:
    vals=n30_df[n30_df.model==mdl].masking_gap.to_numpy()
    print(f"\nCOMPAS {mdl}: mean FMG={vals.mean():.4f}; positive splits={(vals>0).sum()}/{len(vals)}; range=[{vals.min():.4f}, {vals.max():.4f}]")


In [ ]:
# Figure 5: COMPAS extended-seed descriptive distributions
fig, axes = plt.subplots(1,2,figsize=(13,5))
for idx,mdl_name in enumerate(["RF","MLP"]):
    ax=axes[idx]
    vals10=n30_df[(n30_df.model==mdl_name)&(n30_df.seed.isin(SEEDS))].masking_gap.values
    vals30=n30_df[n30_df.model==mdl_name].masking_gap.values
    ax.scatter(np.arange(len(vals10)),vals10,color="#4CAF50",s=48,label=f"first 10 splits (mean={vals10.mean():.3f})")
    vp=ax.violinplot([vals30],positions=[len(vals10)+1.5],showmeans=True,widths=1.8)
    for pc in vp["bodies"]: pc.set_facecolor("#5B9BD5"); pc.set_alpha(.3)
    vp["cmeans"].set_color("#1565C0")
    ax.axhline(vals30.mean(),color="#1565C0",linestyle=":",label=f"30-split mean={vals30.mean():.3f}")
    ax.axhline(0,color="black",linewidth=.7,linestyle=":")
    ax.set_xticks(list(range(len(vals10)))+[len(vals10)+1.5]); ax.set_xticklabels([str(s) for s in SEEDS]+["n=30\nall splits"],fontsize=7,rotation=30)
    ax.set_xlabel("Seed / repeated split"); ax.set_ylabel("FMG at 20% MCAR"); ax.set_title(f"COMPAS {mdl_name}")
    ax.legend(fontsize=7); ax.grid(axis="y",linestyle="--",alpha=.35)
fig.suptitle("COMPAS extended-seed descriptive sensitivity")
plt.tight_layout(); plt.savefig("fig5_compas_extended_seeds.png",bbox_inches="tight",dpi=150); plt.show()


In [ ]:
# Export this notebook run separately; never overwrite the archived release CSVs.
from pathlib import Path
NOTEBOOK_OUTPUT_DIR = Path("fmg_notebook_outputs")
NOTEBOOK_OUTPUT_DIR.mkdir(exist_ok=True)
def save_notebook_output(df, name):
    path=NOTEBOOK_OUTPUT_DIR/name
    df.round(6).to_csv(path,index=False)
    print(f"Saved {path} ({len(df)} rows)")

save_notebook_output(results_df, "results_raw_10seeds.csv")
save_notebook_output(stats_df, "descriptive_primary_summaries.csv")
save_notebook_output(eog_df, "eog_results_10seeds.csv")
save_notebook_output(ablation_df, "imputation_ablation_results.csv")
save_notebook_output(shap_df, "shap_subgroup_results.csv")
save_notebook_output(mar_df, "group_conditional_mar_full_per_seed.csv")
save_notebook_output(mar_stats_df, "group_conditional_mar_full_descriptive_summary.csv")
save_notebook_output(matched_df, "matched_subsample_results.csv")
save_notebook_output(n30_df, "extended_seeds_n30_results.csv")
mg_seed_df=results_df[results_df["rate"]==0.20][["dataset","model","seed","masking_gap","exploratory"]].copy()
save_notebook_output(mg_seed_df,"masking_gap_per_seed.csv")
print("Notebook outputs are separate from the preserved release CSVs.")


In [ ]:
# Verification checklist for descriptive output consistency.
main=stats_df[stats_df["Main"]==True]
expected_rows=len(DATASETS)*len(ALL_MODELS)*(1+len(MISS_RATES))*len(SEEDS)
assert len(results_df)==expected_rows,(len(results_df),expected_rows)
assert len(main)==12,f"Expected 12 primary model-dataset comparisons, found {len(main)}."
assert stats_df[["Dataset","Model"]].duplicated().sum()==0
assert len(mar_df)==len(MAR_DS)*len(MAR_MODELS)*len(SEEDS)*len(MAR_RATES)
assert len(mar_stats_df)==len(MAR_DS)*len(MAR_MODELS)*len(MAR_RATES)
assert main["FMG mean"].notna().all()
print(f"OK raw-result rows: {len(results_df)}/{expected_rows}")
print(f"OK primary model-dataset comparisons: {len(main)}")
print(f"Means above descriptive 0.01 reference: {main['Mean exceeds descriptive 0.01 reference'].sum()}/12")
print(f"Mean FMG across primary comparisons: {main['FMG mean'].mean():.4f}")
print("These checks verify saved benchmark summaries; they do not perform inferential testing.")


In [ ]:
from google.colab import files
import os

# Full list of files generated by the experiment pipeline
all_experimental_files = [
    'results_raw_10seeds.csv',
    'stats_results_10seeds.csv',
    'eog_results_10seeds.csv',
    'imputation_ablation_results.csv',
    'shap_subgroup_results.csv',
    'group_conditional_mar_full_per_seed.csv',
    'group_conditional_mar_full_descriptive_summary.csv',
    'matched_subsample_results.csv',
    'extended_seeds_n30_results.csv',
    'masking_gap_per_seed.csv',
    'fig1_masking_gap_curves.png', 'fig1_masking_gap_curves.pdf',
    'fig2_acc_vs_recall.png', 'fig2_acc_vs_recall.pdf',
    'fig3_mar_vs_mcar.png', 'fig3_mar_vs_mcar.pdf',
    'fig4_class_imbalance.png', 'fig4_class_imbalance.pdf',
    'fig5_compas_extended_seeds.png', 'fig5_compas_extended_seeds.pdf',
    'fig6_shap_shifts.png', 'fig6_shap_shifts.pdf',
    'fig7_imputation_ablation.png', 'fig7_imputation_ablation.pdf',
    'subgroup_denominators_per_seed.csv',
    'masking_diagnostics_10seeds.csv',
    'fmg_bootstrap_ci_primary.csv',
    'fmg_primary_seed_distributions.png',
    'preprocessing_sensitivity_results.csv',
]

print(f"Initiating download for {len(all_experimental_files)} files...")

for filename in all_experimental_files:
    if os.path.exists(filename):
        files.download(filename)
    else:
        print(f"Warning: {filename} not found in the environment.")

## FMG Audit Method (reproducible implementation)
This section implements the Focus-Group Masking Gap audit for an unchanged trained classifier. MCAR masking is applied to test features only, using a training-fitted imputer, with subgroup denominators retained.


In [ ]:
from dataclasses import dataclass

@dataclass
class FMGAuditRecord:
    seed: int
    rate: float
    clean_accuracy: float
    stressed_accuracy: float
    clean_focus_recall: float
    stressed_focus_recall: float
    accuracy_drop: float
    focus_recall_drop: float
    masking_gap: float
    focus_total: int
    focus_positive: int
    focus_negative: int

def fmg_audit(model, X_test, y_test, group_labels, imputer, rates=(0.05,0.10,0.20), seeds=(42,123,456,1,7,13,21,37,55,99), focus_value=0):
    X_test=np.asarray(X_test,dtype=float); y_test=np.asarray(y_test); group_labels=np.asarray(group_labels)
    if len(X_test)!=len(y_test) or len(y_test)!=len(group_labels): raise ValueError('X_test, y_test, and group_labels must have equal length')
    def metrics(X):
        pred=model.predict(X); focus=group_labels==focus_value; yf,pf=y_test[focus],pred[focus]; pos=int((yf==1).sum())
        return dict(accuracy=float(accuracy_score(y_test,pred)),focus_recall=float(recall_score(yf,pf,zero_division=0)),focus_total=int(len(yf)),focus_positive=pos,focus_negative=int(len(yf)-pos))
    clean=metrics(X_test); records=[]
    for seed in seeds:
        for rate in rates:
            rng=np.random.RandomState(int(seed)); X_mask=X_test.copy(); X_mask[rng.rand(*X_mask.shape)<rate]=np.nan
            stressed=metrics(imputer.transform(X_mask)); ad=clean['accuracy']-stressed['accuracy']; rd=clean['focus_recall']-stressed['focus_recall']
            records.append(FMGAuditRecord(int(seed),float(rate),clean['accuracy'],stressed['accuracy'],clean['focus_recall'],stressed['focus_recall'],float(ad),float(rd),float(rd-ad),stressed['focus_total'],stressed['focus_positive'],stressed['focus_negative']))
    return records

def summarize_fmg(records, rate=0.20):
    """Descriptive summaries over repeated benchmark splits; no p-values."""
    vals=np.asarray([r.masking_gap for r in records if r.rate==rate],dtype=float)
    if len(vals)==0: raise ValueError('no records for requested rate')
    return {'n_splits':int(len(vals)),'mean_fmg':float(vals.mean()),
            'sd_fmg':float(np.std(vals,ddof=1)) if len(vals)>1 else 0.0,
            'min_fmg':float(vals.min()),'max_fmg':float(vals.max()),
            'positive_splits':int((vals>0).sum())}


**Usage:** after the existing notebook creates `model`, training-fitted `imp`, transformed `X_te`, `y_te`, and aligned `sens_te`, run `records=fmg_audit(model,X_te,y_te,sens_te,imp)` and `summary=summarize_fmg(records,rate=0.20)`. Export records with `pd.DataFrame([asdict(r) for r in records]).to_csv('fmg_audit_per_seed.csv',index=False)`.


## Descriptive diagnostics and repeated-split variability

The following cells add diagnostics without changing the archived primary metrics. The MCAR rate is a per-feature-cell probability. The notebook records observed cell-level rates and the share of test rows with at least one masked feature. Resampling intervals over the ten split-level outputs are not reported as population confidence intervals because the splits reuse records. No p-values are used to claim population or deployment effects.


In [ ]:
# Explicit MCAR diagnostics and frozen-model checks
def inject_mcar_with_diagnostics(X, rate, seed):
    """Mask test feature cells only and return the mask diagnostics."""
    X_out = X.copy()
    rng = np.random.RandomState(int(seed))
    mask = rng.rand(*X_out.shape) < float(rate)
    X_out[mask] = np.nan
    return X_out, {
        "target_cell_mask_rate": float(rate),
        "observed_cell_mask_rate": float(mask.mean()),
        "row_masked_fraction": float(mask.any(axis=1).mean()),
        "masked_cells": int(mask.sum()),
        "eligible_cells": int(mask.size),
        "test_rows": int(mask.shape[0]),
        "test_features": int(mask.shape[1]),
    }

print("MCAR diagnostics helper defined. A row_masked_fraction is not the target rate;")
print("it is the observed share of test rows containing one or more masked cells.")

diagnostic_rows = []
for ds_name, (df, feats, sens_col, tgt_col, is_expl) in DATASETS.items():
    for seed in SEEDS:
        _, X_te, _, _ = preprocess(df, feats, tgt_col, seed)
        for rate in MISS_RATES:
            _, diag = inject_mcar_with_diagnostics(X_te, rate, seed)
            diagnostic_rows.append({"dataset": ds_name, "seed": seed, "rate": rate,
                                    "exploratory": is_expl, **diag})
masking_diagnostics_df = pd.DataFrame(diagnostic_rows)
masking_diagnostics_df.to_csv("masking_diagnostics_10seeds.csv", index=False)
display(masking_diagnostics_df.head())


In [ ]:
# Seed-level primary distributions and descriptive bootstrap intervals
from pathlib import Path

PRIMARY_DATASETS = ["Adult Income", "COMPAS", "Taiwan Credit"]
PRIMARY_MODELS = ["RF", "XGB", "LGB", "MLP"]
PRIMARY_SEEDS = SEEDS

def bootstrap_mean_ci(values, n_boot=10000, seed=20260909):
    values = np.asarray(values, dtype=float)
    rng = np.random.RandomState(seed)
    samples = rng.choice(values, size=(n_boot, len(values)), replace=True).mean(axis=1)
    return float(values.mean()), float(np.quantile(samples, 0.025)), float(np.quantile(samples, 0.975))

if "masking_gap_per_seed.csv" in [Path(f).name for f in Path('.').glob('masking_gap_per_seed.csv')]:
    _mg = pd.read_csv("masking_gap_per_seed.csv")
else:
    _mg = results_df[results_df["rate"] == 0.20][["dataset", "model", "seed", "masking_gap", "exploratory"]].copy()

_primary = _mg[_mg["dataset"].isin(PRIMARY_DATASETS) & _mg["model"].isin(PRIMARY_MODELS)].copy()
ci_rows = []
for (ds, mdl), sub in _primary.groupby(["dataset", "model"], sort=True):
    mean, lo, hi = bootstrap_mean_ci(sub["masking_gap"].to_numpy())
    ci_rows.append({"dataset": ds, "model": mdl, "n_seeds": len(sub), "mean_fmg": mean,
                    "bootstrap_ci95_low": lo, "bootstrap_ci95_high": hi,
                    "interval_scope": "descriptive repeated-seed benchmark"})
ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv("fmg_bootstrap_ci_primary.csv", index=False)
display(ci_df)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(3, 4, figsize=(15, 9), sharey=True)
for ax, (ds, mdl) in zip(axes.ravel(), [(d, m) for d in PRIMARY_DATASETS for m in PRIMARY_MODELS]):
    vals = _primary[(_primary["dataset"] == ds) & (_primary["model"] == mdl)]["masking_gap"].to_numpy()
    ax.scatter(np.arange(1, len(vals)+1), vals, color="#1677B8", s=28)
    ax.axhline(0, color="black", linewidth=0.7)
    ax.axhline(MG_MIN, color="#C94C4C", linestyle="--", linewidth=0.8)
    ax.set_title(f"{ds}\n{mdl}", fontsize=9)
    ax.set_xlabel("Seed index", fontsize=8)
    ax.set_ylabel("FMG" if mdl == "RF" else "", fontsize=8)
    ax.grid(axis="y", alpha=0.25)
fig.suptitle("Primary seed-level FMG distributions at 20% MCAR", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig("fmg_primary_seed_distributions.png", dpi=180, bbox_inches="tight")
plt.show()


## Type-aware preprocessing sensitivity

This optional sensitivity reruns the stated type-aware pipeline on the three primary datasets, four primary models, ten repeated splits, and one 20% MCAR condition. Results are descriptive for this finite benchmark design. It is not evidence of general preprocessing robustness and does not support seed-based significance claims.

Set `RUN_TYPE_AWARE_SENSITIVITY = True` only when this additional computation is intended; it is heavier than the primary export.


In [ ]:
# Optional in-notebook sensitivity: median numeric / mode categorical only (20% MCAR).
# This cell does NOT calculate KNN/mode or iterative/mode results. Those are generated
# by sensitivity/run_typeaware_imputation.py and reported separately in the manuscript.
RUN_TYPE_AWARE_SENSITIVITY = True

def preprocess_type_aware(df, feats, target, seed):
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.impute import SimpleImputer
    X = df[feats].copy(); y = df[target].values
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.30, random_state=seed, stratify=y)
    cat = list(X_tr.select_dtypes(include=["object", "category"]).columns)
    num = [c for c in feats if c not in cat]
    cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                         ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", MinMaxScaler())])
    pre = ColumnTransformer([("num", num_pipe, num), ("cat", cat_pipe, cat)], remainder="drop")
    Xtr = pre.fit_transform(X_tr); Xte = pre.transform(X_te)
    return pre, X_tr, X_te, y_tr, y_te, X_te.index

def run_type_aware_sensitivity():
    rows = []
    for ds_name in MAIN_DATASETS:
        df, feats, sens_col, tgt_col, _ = DATASETS[ds_name]
        for seed in SEEDS:
            pre, Xtr_raw, Xte_raw, ytr, yte, te_idx = preprocess_type_aware(df, feats, tgt_col, seed)
            sens_te = df.loc[te_idx, sens_col].to_numpy()
            for mdl_name, model in get_models(seed).items():
                if mdl_name == "LR" : continue
                Xtr = pre.transform(Xtr_raw)
                Xte = pre.transform(Xte_raw)
                model.fit(Xtr, ytr)
                clean = evaluate(model, Xte, yte, sens_te)
                clean_prediction_reference = model.predict(Xte).copy()
                rng = np.random.RandomState(seed)
                mask = rng.rand(*Xte_raw.shape) < 0.20
                Xmasked_raw = Xte_raw.copy()
                Xmasked_raw = Xmasked_raw.astype(object)
                Xmasked_raw[mask] = np.nan
                Xmasked = pre.transform(Xmasked_raw)
                stressed = evaluate(model, Xmasked, yte, sens_te)
                assert np.array_equal(clean_prediction_reference, model.predict(Xte)), (
                    "Clean-test predictions changed during masked evaluation.")
                rows.append({"dataset": ds_name, "model": mdl_name, "seed": seed,
                             "rate": 0.20, "masking_gap": masking_gap(clean, stressed),
                             "accuracy_drop": clean["accuracy"]-stressed["accuracy"],
                             "recall_drop": clean["recall_g0"]-stressed["recall_g0"],
                             "preprocessing": "type-aware"})
    out = pd.DataFrame(rows)
    out.to_csv("preprocessing_sensitivity_results.csv", index=False)
    return out

if RUN_TYPE_AWARE_SENSITIVITY:
    preprocessing_sensitivity_df = run_type_aware_sensitivity()
    display(preprocessing_sensitivity_df.head())
else:
    print("Sensitivity run not executed. Set RUN_TYPE_AWARE_SENSITIVITY=True to run it.")


In [ ]:
# Export per-seed subgroup denominators and positive-label counts

denominator_rows = []

for ds_name, (df, feats, sens_col, tgt_col, is_expl) in DATASETS.items():
    for seed in SEEDS:
        _, X_te, _, y_te = preprocess(df, feats, tgt_col, seed)
        sens_te = aligned_test_sensitive(df, X_te, sens_col, y_te)

        focus_mask = sens_te == 0
        y_focus = np.asarray(y_te)[focus_mask]

        denominator_rows.append({
            "dataset": ds_name,
            "seed": int(seed),
            "exploratory": bool(is_expl),
            "focus_group": "sensitive_attribute == 0",
            "subgroup_total": int(len(y_focus)),
            "positive_label_count": int((y_focus == 1).sum()),
            "negative_label_count": int((y_focus == 0).sum()),
            "positive_label_rate": float((y_focus == 1).mean())
                if len(y_focus) else np.nan,
            "recall_denominator": int((y_focus == 1).sum()),
            "no_positive_labels": bool((y_focus == 1).sum() == 0)
        })

subgroup_denominators_df = pd.DataFrame(denominator_rows)
subgroup_denominators_df.to_csv(
    "subgroup_denominators_per_seed.csv",
    index=False
)

display(subgroup_denominators_df)

## Empirical diagnostic-discordance analysis

This additional analysis compares focus-group recall-loss flags with aggregate-accuracy-loss flags at the same illustrative threshold. It tests benchmark-level discordance, not whether an FMG report changes decisions or improves production monitoring. Thresholds are shown as a sensitivity grid and are not operational recommendations. The paired rows are repeated benchmark splits, not independent deployment populations.

For each paired evaluation, let ΔR be clean-minus-masked focus-group recall and ΔA be clean-minus-masked aggregate accuracy. For threshold δ, the analysis records U(δ) = 1[ΔR ≥ δ] × (1 − 1[ΔA ≥ δ]).

Outputs: `fmg_diagnostic_discordance_by_seed.csv`, `fmg_diagnostic_discordance_summary.csv`, and `fmg_diagnostic_discordance_by_dataset_model.csv`.


In [ ]:
# Empirical threshold-discordance analysis: no model is refitted in this cell.
from pathlib import Path
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

DIAGNOSTIC_DATASETS = ["Adult Income", "COMPAS", "Taiwan Credit"]
DIAGNOSTIC_MODELS = ["RF", "XGB", "LGB", "MLP"]
DIAGNOSTIC_RATES = [0.05, 0.10, 0.20]
DIAGNOSTIC_THRESHOLDS = [0.01, 0.025, 0.05, 0.10]

if "results_df" not in globals() or not isinstance(results_df, pd.DataFrame):
    candidates = [Path("results_raw_10seeds.csv"), Path("/content/results_raw_10seeds.csv")]
    result_path = next((p for p in candidates if p.is_file()), None)
    if result_path is None:
        raise FileNotFoundError(
            "Run the primary pipeline first or place results_raw_10seeds.csv in the current folder."
        )
    diagnostic_results = pd.read_csv(result_path)
else:
    diagnostic_results = results_df.copy()

needed = {"dataset", "model", "seed", "rate", "accuracy", "recall_g0", "masking_gap"}
missing = needed.difference(diagnostic_results.columns)
if missing:
    raise ValueError(f"Missing result columns: {sorted(missing)}")

primary = diagnostic_results[
    diagnostic_results["dataset"].isin(DIAGNOSTIC_DATASETS)
    & diagnostic_results["model"].isin(DIAGNOSTIC_MODELS)
    & diagnostic_results["rate"].isin([0.0, *DIAGNOSTIC_RATES])
].copy()
clean = primary.loc[
    primary.rate == 0, ["dataset", "model", "seed", "accuracy", "recall_g0"]
].rename(columns={"accuracy": "accuracy_clean", "recall_g0": "recall_clean"})
paired = primary.loc[primary.rate > 0].merge(
    clean, on=["dataset", "model", "seed"], how="inner", validate="many_to_one"
)
expected_rows = len(DIAGNOSTIC_DATASETS) * len(DIAGNOSTIC_MODELS) * len(DIAGNOSTIC_RATES) * 10
assert len(paired) == expected_rows, f"Expected {expected_rows} paired rows, found {len(paired)}"
paired["recall_loss"] = paired["recall_clean"] - paired["recall_g0"]
paired["accuracy_loss"] = paired["accuracy_clean"] - paired["accuracy"]
paired["fmg_recomputed"] = paired["recall_loss"] - paired["accuracy_loss"]
assert np.allclose(paired["fmg_recomputed"], paired["masking_gap"], atol=2.1e-6, rtol=0)

records = []
for row in paired.itertuples(index=False):
    for delta in DIAGNOSTIC_THRESHOLDS:
        recall_flag = int(row.recall_loss >= delta)
        accuracy_flag = int(row.accuracy_loss >= delta)
        records.append({
            "dataset": row.dataset,
            "model": row.model,
            "seed": int(row.seed),
            "masking_rate": float(row.rate),
            "threshold": float(delta),
            "recall_loss": float(row.recall_loss),
            "accuracy_loss": float(row.accuracy_loss),
            "fmg": float(row.fmg_recomputed),
            "recall_loss_flag": recall_flag,
            "accuracy_loss_flag": accuracy_flag,
            "accuracy_only_miss": int(recall_flag == 1 and accuracy_flag == 0),
            "both_flagged": int(recall_flag == 1 and accuracy_flag == 1),
        })
diagnostic_detail = pd.DataFrame(records)

diagnostic_detail["neither_flagged"] = (
    (diagnostic_detail.recall_loss_flag == 0) & (diagnostic_detail.accuracy_loss_flag == 0)
).astype(int)
summary_aggregations = {
    "n_paired_evaluations": ("seed", "size"),
    "recall_loss_flags": ("recall_loss_flag", "sum"),
    "accuracy_loss_flags": ("accuracy_loss_flag", "sum"),
    "accuracy_only_misses": ("accuracy_only_miss", "sum"),
    "both_flagged": ("both_flagged", "sum"),
    "neither_flagged": ("neither_flagged", "sum"),
}
diagnostic_summary = diagnostic_detail.groupby(
    ["masking_rate", "threshold"], sort=True
).agg(**summary_aggregations).reset_index()
diagnostic_by_dataset_model = diagnostic_detail.groupby(
    ["dataset", "model", "masking_rate", "threshold"], sort=True
).agg(**summary_aggregations).reset_index()
for _frame in (diagnostic_summary, diagnostic_by_dataset_model):
    _frame["missed_recall_flags_pct"] = np.where(
        _frame.recall_loss_flags > 0,
        100 * _frame.accuracy_only_misses / _frame.recall_loss_flags,
        np.nan,
    )
_count_columns = [
    "n_paired_evaluations", "recall_loss_flags", "accuracy_loss_flags",
    "accuracy_only_misses", "both_flagged", "neither_flagged",
]
diagnostic_summary[_count_columns] = diagnostic_summary[_count_columns].astype("int64")
diagnostic_by_dataset_model[_count_columns] = diagnostic_by_dataset_model[_count_columns].astype("int64")
diagnostic_output = Path("fmg_diagnostic_value_outputs")
diagnostic_output.mkdir(exist_ok=True)
diagnostic_detail.to_csv(diagnostic_output / "fmg_diagnostic_discordance_by_seed.csv", index=False)
diagnostic_summary.to_csv(diagnostic_output / "fmg_diagnostic_discordance_summary.csv", index=False)
diagnostic_by_dataset_model.to_csv(
    diagnostic_output / "fmg_diagnostic_discordance_by_dataset_model.csv", index=False
)

print("Descriptive paired threshold comparison; no inferential p-values or deployment claims.")
display(diagnostic_summary)
display(diagnostic_summary[diagnostic_summary.masking_rate == 0.20])


In [ ]:
# Run manifest: provenance for this execution, written only after the full run.
import hashlib as _hashlib
import importlib.metadata as _metadata
import platform as _platform
import sys as _sys
import os as _os

def _file_sha256(path):
    h = _hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def _frame_sha256(frame):
    values = pd.util.hash_pandas_object(frame, index=True).values.tobytes()
    return _hashlib.sha256(values).hexdigest()

def _source_record(source, frame):
    p = Path(str(source))
    return {
        "source": str(source),
        "file_sha256": _file_sha256(p) if p.is_file() else None,
        "rows_after_dataset_cleaning": int(len(frame)),
        "columns_after_dataset_cleaning": int(len(frame.columns)),
        "cleaned_frame_sha256": _frame_sha256(frame),
    }

_dataset_records = {}
for _name, (_frame, _features, _sensitive, _target, _exploratory) in DATASETS.items():
    _source = {
        "Adult Income": globals().get("ADULT_SOURCE", "UCI Adult URL"),
        "COMPAS": "ProPublica COMPAS analysis CSV URL",
        "Taiwan Credit": globals().get("TAIWAN_SOURCE", "user-uploaded CSV path not recorded"),
        "German Credit": "OpenML data_id=31 (credit-g)",
    }.get(_name, "source not specified")
    _dataset_records[_name] = _source_record(_source, _frame)
    _dataset_records[_name]["exploratory"] = bool(_exploratory)

if "Adult Income" in _dataset_records:
    _dataset_records["Adult Income"]["raw_rows"] = int(globals().get("ADULT_RAW_ROWS", -1))
    _dataset_records["Adult Income"]["rows_with_missing"] = int(globals().get("ADULT_MISSING_ROWS", -1))

_output_paths = sorted([p for p in NOTEBOOK_OUTPUT_DIR.iterdir()
                        if p.is_file() and p.name != "run_manifest.json"])
_output_paths += sorted([p for p in Path(".").glob("fig*.png") if p.is_file()])
_output_paths += sorted([p for p in Path(".").glob("fig*.pdf") if p.is_file()])
_manifest = {
    "status": "completed",
    "scope": "fresh notebook run; repeated seeds are benchmark splits, not independent populations",
    "runtime_seconds_from_primary_start": float(time.time() - start_time),
    "environment": {
        "python": _sys.version,
        "platform": _platform.platform(),
        "processor_reported": _platform.processor(),
        "logical_cpu_count": _os.cpu_count(),
        "packages": {name: _metadata.version(name) for name in
                     ["numpy", "pandas", "scipy", "scikit-learn", "xgboost", "lightgbm", "shap", "matplotlib", "seaborn"]},
    },
    "design": {"seeds": list(SEEDS), "mcar_rates": list(MISS_RATES),
               "models": list(ALL_MODELS), "primary_datasets": list(MAIN_DATASETS)},
    "adult_parser": {"missing_markers": ["?", " ?"], "complete_case_rows": 30162,
                     "observed_raw_rows": int(globals().get("ADULT_RAW_ROWS", -1)),
                     "observed_missing_rows": int(globals().get("ADULT_MISSING_ROWS", -1))},
    "datasets": _dataset_records,
    "outputs": {p.name: {"sha256": _file_sha256(p), "bytes": p.stat().st_size} for p in _output_paths},
}
manifest_path = NOTEBOOK_OUTPUT_DIR / "run_manifest.json"
manifest_path.write_text(json.dumps(_manifest, indent=2), encoding="utf-8")
print(f"Run manifest saved: {manifest_path}")
print(f"Elapsed seconds: {_manifest['runtime_seconds_from_primary_start']:.1f}")
print("This manifest describes this run only; it does not make the repeated seeds independent populations.")
